# 2.0 Preprocessing 

### Feb 2026

- Noticed issue with ppm to pco2 atmospheric conversion
- Now have sections for processing carbon variables for each platform 


### Jan 2026
Combines what was previously 2.0 and 2.1 into a unified notebook to run 

Output is in ```../working-vars/regression/inputs/<pc_params>```


2.1_process_pco2_data 
- Start with clustered, QC'ed open ocean data from section 1. 
Convert all data to pCO2 (choose which correction to use for floats, convert fco2 to pco2 for socat)
- Add MLD to coreDS
- Add seasonal time
- Save as regression inputs 


***Note that delta-pco2 is OCEAN - ATMOSPHERE***

In [2]:
import mod_loading as loader
from importlib import reload
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime
import gsw
import mod_argo
import matplotlib.pyplot as plt
from cmocean import cm as cmo


# === CUSTOM FUNCTIONS
import mod_preprocessing as mod_prep
import mod_ocean


# Import data


- Now doing this before clustering

In [ ]:
# The output of this notebook can be accessed:
# [coreDS, coreINDEX] = loader.import_core_data(type='processed') # has adt, sla, mld, pco2

In [ ]:
# To start new, 
[coreDS, coreINDEX] = loader.import_core_data(type='L3_only')
# coreINDEX, coreDS = mod_prep.add_mixedlayer_pressure(coreINDEX, coreDS) # add mld

[bgcDS, bgcINDEX] = loader.import_bgc_data(type = 'L3_only')

In [ ]:
# [coreDS, coreINDEX] = loader.import_core_data(type='L3_only')
# [coreDS, coreINDEX] = loader.import_core_data(type='processed')
# Marine boundary layer atmospheric CO2 data
# mbl_co2 = pd.read_csv('../data/marineboundarylayer/co2_mbl_2014-2018_data_tabular.csv', index_col=0)
# mbl_co2 = xr.open_dataset('../data/marineboundarylayer/co2_mbl_2014-2023_dataset_combined.nc')

In [ ]:
# outdated. now MLD addition done earlier and already in notebook 
# Add MLD
reload(mod_prep)
# bgcINDEX, bgcDS = mod_prep.add_mixedlayer_pressure(bgcINDEX, bgcDS)
# coreINDEX, coreDS = mod_prep.add_mixedlayer_pressure(coreINDEX, coreDS)

# To access the output in another notebook, run:
# [bgcDS, bgcINDEX] = loader.import_bgc_data(L3_only=False)
# datetag = '20260121'
# coreINDEX = xr.open_dataset('../working-vars/argo/import-L3/coreINDEX_clustered_pc8_gmm6_2014-2023_withMLD_acc' + datetag + '.nc')
# coreDS = xr.open_dataset('../working-vars/argo/import-L3/coreDATA_clustered_pc8_gmm6_2014-2023_withMLD_acc' + datetag + '.nc')


/opt/homebrew/Caskroom/mambaforge/base/envs/cremas/lib/python3.9/site-packages/xarray/coding/times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(


# Train/Val: Process SOCAT fCO2 to pCO2

- Convert fCO2 to pCO2 using pyCO2sys
- Add atmospheric sea level pressure

In [ ]:
# If rerunning, this function gives back the shipDF with processed fco2
# [_, shipDF] = loader.import_processed_inputs()

In [55]:
# If starting new, import socat colocation data
socat_colocated = loader.import_socat_colocation() # 7d 
socat_df = socat_colocated[socat_colocated['latitude']<-35]
socat_df

,longitude,latitude,fco2rec,sal,sst,yearday,fco2water_equ_wet,fco2water_sst_wet,pco2water_equ_wet,pco2water_sst_wet,...,xco2water_sst_dry,datetime,expoID,bathymetry,nearest_profid,prof_datetime,prof_lat,prof_lon,yd_sep,km_sep
0,-24.64215,-73.725900,341.6865,34.074,-1.2305,1.483947,NaN,341.700,NaN,343.200,...,NaN,2014-01-02 11:36:53,06AQ20131221_id0,-2062.0,7900378_id041,2014-01-09,-71.616580,-22.823985,6.720278,242.124951
1,-36.40135,-77.100350,365.8905,34.281,-1.7040,9.499502,NaN,365.900,NaN,367.500,...,NaN,2014-01-10 11:59:17,06AQ20131221_id0,-1087.0,7900378_id041,2014-01-09,-71.616580,-22.823985,1.295278,729.272575
2,-38.14620,-77.899100,275.3920,34.279,-1.4480,15.491701,NaN,275.400,NaN,276.600,...,NaN,2014-01-16 11:48:03,06AQ20131221_id0,-1168.0,7900378_id042,2014-01-19,-71.682778,-23.213771,2.700012,811.679788
3,-38.94590,-77.608000,375.1900,34.418,-1.7700,16.295197,NaN,375.200,NaN,376.900,...,NaN,2014-01-17 07:05:05,06AQ20131221_id0,-1009.0,7900378_id042,2014-01-19,-71.682778,-23.213771,1.896516,799.773551
4,-28.47890,-74.557400,298.7880,33.760,-1.3940,21.498843,NaN,298.800,NaN,300.100,...,NaN,2014-01-22 11:58:20,06AQ20131221_id1,-1639.0,7900378_id043,2014-01-29,-71.743994,-23.090656,6.695984,357.493891
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6283,163.00795,-39.536095,370.5850,35.444,15.0160,3497.534375,372.560,370.135,NaN,371.470,...,377.375,2023-07-30 12:49:30,PAT520230724_id1,-3646.0,5905508_id071,2023-08-01,-39.699600,163.253800,2.079502,27.820801
6284,154.23600,-37.704690,364.7370,35.543,16.0950,3498.604861,366.830,364.440,NaN,365.740,...,370.690,2023-07-31 14:31:00,PAT520230724_id1,-4523.0,5906152_id108,2023-07-30,-37.819290,152.754590,1.312465,130.846019
6285,149.00550,-38.579580,366.4580,35.644,15.7330,3499.499306,368.570,366.130,NaN,367.450,...,367.770,2023-08-01 11:59:00,PAT520230724_id1,-2813.0,5905515_id120,2023-07-28,-38.527290,150.643725,3.723206,142.572572
6286,177.57530,-37.193580,350.2155,35.146,15.6535,3574.664236,352.555,349.850,NaN,351.125,...,354.565,2023-10-15 15:56:30,PAT520231011_id1,-1709.0,5906421_id087,2023-10-15,-37.191630,178.572070,0.086331,88.292512


In [56]:
# ==== Convert SOCAT fugacity to partial pressure pCO2 (~1 min)
# Updated Feb 06 2026 # Jan 21 2026

shipDF = mod_prep.convert_socat_fco2(socat_df)
shipDF['mld'] = shipDF.nearest_profid.apply(lambda x: coreINDEX.sel(profid=x).mld.values)

shipDF = mod_prep.add_regression_time_vars(shipDF)
shipDF = mod_prep.add_regression_carbon_vars(shipDF, convert_pco2=True) 
shipDF['delta_pco2'] = shipDF['pco2_ocean'] - shipDF['pco2_atmos']


/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_ocean.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['datetime'].dt.is_leap_year.replace({True: 366, False: 365}))


In [66]:
# Add ADT and add profid to soccom_df
print('Adding satellite ADT...')
shipDF_adt = mod_prep.add_satellite_adt(shipDF, year_range = range(2014,2024))

Adding satellite ADT...


In [68]:
use_cols = [ 'cruiseid', 'datetime',  # 'sid', 'cluster', 
    'longitude', 'latitude', 'linear_time',
    'nearest_profid', 'prof_datetime', 'prof_lat', 'prof_lon', 'yd_sep', 'km_sep', 
    'ydcos', 'ydsin',
    'CT', 'SA', 'sst', 'sss', 'mld',  
    'adt', #'pressure', 
    'atmos_pres_Pa', 'atmos_pres_atm',  'vapor_pres_atm',
    'fco2rec', 
    'pco2_atmos', 'pco2_ocean', 'delta_pco2']

shipDF_processed = (shipDF_adt.rename(columns={'cruiseID':'cruiseid'})
                                .reset_index()[use_cols].copy())

print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
shipDF_processed


2026-02-11 13:43:31


,cruiseid,datetime,longitude,latitude,linear_time,nearest_profid,prof_datetime,prof_lat,prof_lon,yd_sep,...,sss,mld,adt,atmos_pres_Pa,atmos_pres_atm,vapor_pres_atm,fco2rec,pco2_atmos,pco2_ocean,delta_pco2
0,06AQ20131221,2014-01-02 11:36:53,-24.64215,-73.725900,1.483947,7900378_id041,2014-01-09,-71.616580,-22.823985,6.720278,...,34.405349,11.667522,-1.2091,99810.0,0.985048,0.005405,341.6865,385.975982,343.219907,-42.756075
1,06AQ20131221,2014-01-10 11:59:17,-36.40135,-77.100350,9.499502,7900378_id041,2014-01-09,-71.616580,-22.823985,1.295278,...,34.614034,11.667522,-0.9939,99620.0,0.983173,0.005219,365.8905,385.238844,367.543164,-17.695680
2,06AQ20131221,2014-01-16 11:48:03,-38.14620,-77.899100,15.491701,7900378_id042,2014-01-19,-71.682778,-23.213771,2.700012,...,34.612038,10.548086,-0.9822,99320.0,0.980212,0.005318,275.3920,383.955134,276.631562,-107.323573
3,06AQ20131221,2014-01-17 07:05:05,-38.94590,-77.608000,16.295197,7900378_id042,2014-01-19,-71.682778,-23.213771,1.896516,...,34.752420,10.548086,-1.0041,99440.0,0.981396,0.005193,375.1900,384.469845,376.886195,-7.583650
4,06AQ20131221,2014-01-22 11:58:20,-28.47890,-74.557400,21.498843,7900378_id043,2014-01-29,-71.743994,-23.090656,6.695984,...,34.088126,10.657734,-1.2278,101040.0,0.997187,0.005341,298.7880,390.543232,300.131879,-90.411353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6283,PAT520230724,2023-07-30 12:49:30,163.00795,-39.536095,3497.534375,5905508_id071,2023-08-01,-39.699600,163.253800,2.079502,...,35.779323,253.554480,0.7501,101420.0,1.000938,0.016504,370.5850,410.441585,371.927784,-38.513801
6284,PAT520230724,2023-07-31 14:31:00,154.23600,-37.704690,3498.604861,5906152_id108,2023-07-30,-37.819290,152.754590,1.312465,...,35.879350,154.492616,0.5157,101750.0,1.004194,0.017686,364.7370,411.361558,366.040624,-45.320934
6285,PAT520230724,2023-08-01 11:59:00,149.00550,-38.579580,3499.499306,5905515_id120,2023-07-28,-38.527290,150.643725,3.723206,...,35.981165,123.963626,0.6250,101740.0,1.004096,0.017280,366.4580,411.463255,367.773796,-43.689459
6286,PAT520231011,2023-10-15 15:56:30,177.57530,-37.193580,3574.664236,5906421_id087,2023-10-15,-37.191630,178.572070,0.086331,...,35.478500,105.351299,0.6976,102440.0,1.011004,0.017197,350.2155,415.097406,351.474244,-63.623161


In [196]:
# foo = mod_prep.add_satellite_adt(shipDF_processed, year_range = range(2014,2024))
# foo

# platDF, year_range = range(2014,2024)):
#     platDF =  mod_ocean.expand_datetime(platDF, type='dataframe')
#     platDF_annual = {k:None for k in year_range} # store by year

#     adt_dict = {k:None for k in year_range}
#     for open_yr in adt_dict.keys():
#         folder = '/Volumes/crusoe-repo/data/copernicusmarine/' # used to be in OneDrive/Code/CRUSOE
#         filepath = ('cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D_adt-sla_179.94W-179.94E_88.94S-35.06S_' + str(open_yr) + '-01-01-' + str(open_yr) + '-12-31.nc')
#         adt_dict[open_yr] = xr.open_dataset(folder + filepath)

#     for yr in year_range:
#         platDF_annual[yr] = platDF[platDF.year==yr]
#         adt_data = adt_dict[yr]
#         platDF_annual[yr]['adt'] =  platDF_annual[yr].apply(lambda row: get_row_adt(row, adt_data), axis=1)

#     # Separate into ship and float
#     platDF_added = pd.concat(platDF_annual.values(), axis=0)


/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_preprocessing.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  platDF_annual[yr]['adt'] =  platDF_annual[yr].apply(lambda row: get_row_adt(row, adt_data), axis=1)
/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_preprocessing.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  platDF_annual[yr]['adt'] =  platDF_annual[yr].apply(lambda row: get_row_adt(row, adt_data), axis=1)
/Users/sangminsong/Library/CloudSt

RuntimeError: NetCDF: HDF error

In [11]:
# result = shipDF_processed[['atmos_co2_ppm', 'pco2_atmos']].head(1000)
# result['percent_diff'] = result.apply(lambda row: (row.pco2_atmos - row.atmos_co2_ppm)/row.atmos_co2_ppm * 100, axis=1)
# result.describe()

,pco2_atmos,percent_diff
count,1000.000000,1000.000000
mean,398.006527,-1.667455
std,6.022208,1.184274
min,376.632009,-5.207693
25%,394.151272,-2.460089
50%,398.526932,-1.624621
75%,402.276366,-0.774142
max,413.089665,1.608417


In [11]:
# ==== Optional Save  # better to save after adding adt
# save=True
# datetag = datetime.now().strftime('%Y%m%d')

# # Choose final variables
# final_ship = shipDF_processed.rename(columns={'expoID': 'sampleid', 'cruiseID':'cruiseid'})
# save_cols = ['sampleid', 'cruiseid', 'cluster', 'nearest_profid',
#         'datetime', 'latitude', 'longitude', 
#         'linear_time', 'ydcos', 'ydsin', 'decimalyr', 
#         'CT', 'SA', 'mld', 'atm_pres_hPa',
#         'fco2rec', 'pco2', 
#         'pco2_atm', 'delta_pco2']
# shipDF_final = final_ship[save_cols]

# if save:
#     savepath = '../working-vars/regression/inputs/'
#     shipDF_final.to_csv(savepath + 'shipDF_socatv2024_1d_pCO2converted_co2sys_PCM_8pc8class_acc' + datetag + '.csv')

#     print('Saved ship data to ' + savepath + ', datetag ' + datetag)

Saved ship data to ../working-vars/regression/inputs/, datetag 20260121


# Train/Val: Process SOCCOM float corrected pCO2 (surface 20m averages)

In [7]:
# ===UDPATED FEB 10 = Make training data: Process corrected pH to pCO2 data from 20m averages using bgcDS
# ~ 1 min run time for 10 year dataset
# Feb 10 changing to include 2024 data. split later to separate out test data 

use_float_var = 'pCO2_pHbias5_pK1'
print('Using corrected variable: <' + use_float_var + '> for float pCO2')

soccom_df = mod_prep.process_soccom_20m_averages(start_date = '2013-12-31', end_date = '2024-12-31', use_float_var='pCO2_pHbias5_pK1')

print('Adding regression vars...')
soccom_df = mod_prep.add_regression_time_vars(soccom_df)
soccom_df = mod_prep.add_regression_carbon_vars(soccom_df, convert_pco2=True) 
soccom_df['delta_pco2'] = soccom_df['pco2_ocean'] - soccom_df['pco2_atmos']

print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Using corrected variable: <pCO2_pHbias5_pK1> for float pCO2
Imported SOCCOM pCO2 data using variable pCO2_pHbias5_pK1
Adding regression vars...


/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_ocean.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['datetime'].dt.is_leap_year.replace({True: 366, False: 365}))


2026-02-11 13:10:54


In [ ]:
# soccom_df.to_csv('../working-vars/regression/inputs/temp_soccomdf_beforeadt.csv')

In [51]:
# Add ADT and add profid to soccom_df
finderINDEX = xr.merge([bgcINDEX, bgcINDEX_test]) # combine ten-year and 2024 together
soccomDF_processed = mod_prep.match_profid_from_datetag(soccom_df.reset_index(), finderINDEX)

print('Adding satellite ADT...')
soccomDF_processed = mod_prep.add_satellite_adt(soccomDF_processed, year_range = range(2014,2025))


Adding satellite ADT...


In [87]:
# === Finalize columns
use_cols = ['wmoid', 'profid', 'prof_datetag', 'datetime', #'cluster'
        'latitude', 'longitude', 'ydcos', 'ydsin',
        'CT', 'SA',  'sst', 'sss', 'mld', 
        'adt',
        'atmos_pres_Pa', 'atmos_pres_atm',  'vapor_pres_atm',
        'pCO2_standard', 'pCO2_pHbias3', 'pCO2_pHbias5', 'pCO2_pHbias3_pK1', 'pCO2_pHbias5_pK1', 
        'pco2_atmos', 'pco2_ocean', 'delta_pco2']

soccomDF_processed = soccomDF_processed.reset_index()[use_cols]

# Split up ten year period 2014-2023 (train,val) and 2024 (test)
floatDF_processed = soccomDF_processed[soccomDF_processed.datetime.dt.year < 2024]
testDF_processed = soccomDF_processed[soccomDF_processed.datetime.dt.year == 2024]
testDF_processed['profid'] = ([x.replace('id', 'testid') for x in testDF_processed['profid']])


print(len(floatDF_processed), 'profiles in training/validation set, from 2014-2023')
print(len(testDF_processed), 'profiles in test set, from 2024')

print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

10821 profiles in training/validation set, from 2014-2023
2113 profiles in test set, from 2024
2026-02-11 14:08:46


/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_87096/4205016496.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  testDF_processed['profid'] = ([x.replace('id', 'testid') for x in testDF_processed['profid']])


In [88]:
floatDF_processed

,wmoid,profid,prof_datetag,datetime,latitude,longitude,ydcos,ydsin,CT,SA,...,atmos_pres_atm,vapor_pres_atm,pCO2_standard,pCO2_pHbias3,pCO2_pHbias5,pCO2_pHbias3_pK1,pCO2_pHbias5_pK1,pco2_atmos,pco2_ocean,delta_pco2
0,5904187,5904187_id002,5904187_20140424,2014-04-24 01:35:59.999999,-45.027,209.980,-0.365551,0.930791,13.499859,34.489418,...,0.984851,0.014965,371.755291,368.766597,366.786132,361.531042,359.594018,382.385383,359.594018,-22.791365
1,5904187,5904187_id003,5904187_20140429,2014-04-29 10:21:59.999998,-44.925,209.465,-0.449781,0.893139,13.255945,34.504120,...,1.009524,0.014729,368.366839,365.405713,363.443515,358.234974,356.315818,392.294217,356.315818,-35.978399
2,5904187,5904187_id004,5904187_20140504,2014-05-04 17:40:00.000001,-45.037,209.571,-0.529291,0.848440,13.192600,34.513058,...,1.003898,0.014668,370.674124,367.697147,365.724439,360.473817,358.544402,390.099938,358.544402,-31.555536
3,5904187,5904187_id005,5904187_20140510,2014-05-10 00:45:59.999996,-44.769,209.377,-0.604283,0.796770,13.069445,34.476292,...,0.994720,0.014550,372.994354,370.002674,368.020214,362.723038,360.784113,386.600808,360.784113,-25.816695
4,5904187,5904187_id006,5904187_20140515,2014-05-15 07:09:00.000005,-44.989,209.152,-0.673884,0.738837,12.586998,34.523502,...,1.012485,0.014099,367.886035,364.937159,362.983059,357.751598,355.840425,393.854081,355.840425,-38.013657
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10816,5906583,5906583_id030,5906583_20231110,2023-11-10 05:56:00.000004,-62.603,163.310,0.622491,-0.782627,-1.736849,34.314301,...,0.989983,0.005205,436.629351,433.365990,431.203083,424.171180,422.057030,411.476586,422.057030,10.580444
10817,5906583,5906583_id031,5906583_20231120,2023-11-20 12:59:00.000000,-62.689,162.887,0.750620,-0.660735,-1.672186,34.215834,...,1.001431,0.005230,422.934803,419.767349,417.668016,410.878138,408.826105,416.243436,408.826105,-7.417331
10818,5906583,5906583_id032,5906583_20231130,2023-11-30 19:55:59.999998,-62.720,162.534,0.855236,-0.518240,-1.683684,34.142183,...,0.988305,0.005226,415.820134,412.703188,410.637335,403.970936,401.951619,410.705222,401.951619,-8.753604
10819,5906583,5906583_id033,5906583_20231211,2023-12-11 04:32:00.000003,-62.700,162.647,0.933550,-0.358446,-1.182184,33.978353,...,0.981100,0.005423,412.723012,409.624079,407.570172,400.971131,398.963470,407.573351,398.963470,-8.609881


In [89]:
testDF_processed

,wmoid,profid,prof_datetag,datetime,latitude,longitude,ydcos,ydsin,CT,SA,...,atmos_pres_atm,vapor_pres_atm,pCO2_standard,pCO2_pHbias3,pCO2_pHbias5,pCO2_pHbias3_pK1,pCO2_pHbias5_pK1,pco2_atmos,pco2_ocean,delta_pco2
10821,1902494,1902494_testid005,1902494_20240414,2024-04-14 01:05:59.999995,-46.051,8.418,-0.208893,0.977938,8.362248,33.980043,...,0.997977,0.010632,385.186597,382.174335,380.178073,374.436772,372.484853,412.286438,372.484853,-39.801585
10822,1902494,1902494_testid006,1902494_20240415,2024-04-15 00:34:59.999996,-46.079,8.489,-0.225324,0.974284,8.462328,33.977941,...,0.996990,0.010704,390.867094,387.813215,385.789368,379.953823,377.974947,411.843348,377.974947,-33.868401
10823,1902494,1902494_testid006,1902494_20240415,2024-04-15 23:15:59.999996,-46.119,8.556,-0.241134,0.970492,8.386343,33.973935,...,0.999457,0.010649,390.203562,387.155400,385.135341,379.307806,377.332637,412.895518,377.332637,-35.562880
10824,1902494,1902494_testid008,1902494_20240416,2024-04-16 21:53:00.000002,-46.144,8.612,-0.256834,0.966456,8.339002,33.972940,...,1.001036,0.010615,387.736860,384.706857,382.698834,376.911902,374.948496,413.568473,374.948496,-38.619977
10825,1902494,1902494_testid009,1902494_20240417,2024-04-17 20:03:00.000003,-46.172,8.652,-0.272156,0.962253,8.574900,33.975963,...,1.000345,0.010786,390.031218,386.981940,384.961146,379.144770,377.168868,413.207793,377.168868,-36.038925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12929,7902134,7902134_testid001,7902134_20241202,2024-12-02 05:03:00.000003,-43.610,266.023,0.873645,-0.486564,11.344387,34.209100,...,0.997681,0.012986,421.150480,417.841200,415.648130,409.427093,407.282639,411.264266,407.282639,-3.981627
12930,7902134,7902134_testid002,7902134_20241212,2024-12-12 10:44:59.999997,-43.667,266.415,0.945378,-0.325975,12.071960,34.221409,...,1.004392,0.013626,460.583094,456.982503,454.596343,447.729779,445.396625,413.797331,445.396625,31.599294
12931,7902134,7902134_testid003,7902134_20241222,2024-12-22 16:30:59.999996,-43.583,266.845,0.987875,-0.155254,12.017707,34.200234,...,1.005774,0.013577,474.052716,470.357643,467.908847,460.804132,458.409796,414.398989,458.409796,44.010807
12932,7902133,7902133_testid002,7902133_20241207,2024-12-07 07:07:59.999998,-49.189,250.268,0.912825,-0.408351,8.931264,34.281195,...,0.963731,0.011051,456.924390,453.393309,451.053139,444.097836,441.809882,397.741042,441.809882,44.068841


In [9]:
# ===outdated, but works = Make training data: Process corrected pH to pCO2 data from 20m averages using bgcDS
# ~ 1 min run time for 10 year dataset
# [bgcINDEX, bgcDS, socatDS] = loader.import_clustered_data(pcm_params=pcm_params)

# Updated Feb 06 2026
# use_var = 'pCO2_pHbias5_pK1'
# print('Using corrected variable: <' + use_var + '> for float pCO2')
# soccom_df = mod_prep.process_soccom_20m_clusters(bgcDS, use_var=use_var,
#                                                  start_date = '2013-12-31', end_date = '2023-12-31') # using new clustered bgcDS
# soccom_df['pco2_ocean'] = soccom_df[use_var]

# soccom_df = mod_prep.add_regression_time_vars(soccom_df)
# soccom_df = mod_prep.add_regression_carbon_vars(soccom_df, convert_pco2=True) 
# soccom_df['delta_pco2'] = soccom_df['pco2_ocean'] - soccom_df['pco2_atmos']

# use_cols = ['wmoid', 'profid', 'prof_datetag', 'cluster', 'datetime', 
#         'latitude', 'longitude', 'ydcos', 'ydsin',
#         'CT', 'SA', 'mld',  'sst', 'sss',
#         'atmos_pres_Pa', 'atmos_pres_atm',  'vapor_pres_atm',
#         'pCO2_standard', 'pCO2_pHbias3', 'pCO2_pHbias5', 'pCO2_pHbias3_pK1', 'pCO2_pHbias5_pK1', 
#         'pco2_atmos', 'pco2_ocean', 'delta_pco2']

# floatDF_processed = soccom_df.reset_index()[use_cols].copy()

Using corrected variable: <pCO2_pHbias5_pK1> for float pCO2
Clusters assigned.
Initially missing clusters for 623 profiles, filling in ==>
Filled in 296 missing clusters using adjacent profiles


In [13]:
# === OPTIONAL SAVE # better to save after adding adt
# save = True
# savepath = '../working-vars/regression/inputs/'
# float_parameter = 'pHbias5_pK1'
# datetag = datetime.now().strftime('%Y%m%d')

# # Choose final variables
# save_cols = ['profid', 'wmoid', 'prof_datetag', 'cluster', 
#             'datetime', 'latitude', 'longitude', 
#             'linear_time', 'ydcos', 'ydsin', 'decimalyr', 
#             'CT', 'SA', 'mld', 
#             'pCO2_standard', 'pCO2_pHbias3',
#             'pCO2_pHbias5', 'pCO2_pHbias3_pK1', 'pCO2_pHbias5_pK1' ,'pco2', 
#             'pco2_atm', 'delta_pco2']
# floatDF_final = floatDF_processed[save_cols]

# if save:
#     floatDF_final.to_csv(savepath + 'floatDF_soccom20m_pco2_' + float_parameter + '_PCM_8pc8class_acc' + datetag + '.csv')
#     print('Saved float data to ' + savepath + ', datetag ' + datetag)

Saved float data to ../working-vars/regression/inputs/, datetag 20260121


# SAVE P1-processed: ship and float data

p1-processed has carbon variable, mld, adt

In [ ]:
# use_float_var = 'pCO2_pHbias5_pK1'

In [95]:
# === Optional save: 
# newest version does preprocessing before clustering step 
# all variables at "P1" stage should have mld, adt, atmospheric pco2, deltapco2 all calculated

save = True
datetag = datetime.now().strftime('%Y%m%d') 

if save: # new version as of feb 2026 
    filepath = '../working-vars/regression/P1-processed/'

    # Floats
    filename = 'bgcArgo_trainval_processed_co2_mld_adt_soccom20m_' + use_float_var + '_yr2014-2023_acc' + datetag + '.csv'
    floatDF_processed.to_csv(filepath + filename)
    print('Saved bgcArgo training/validation data to ' + filepath + filename)

    filename = 'bgcArgo_test_processed_co2_mld_adt_soccom20m' + use_float_var + '_yr2024_acc' + datetag + '.csv'
    testDF_processed.to_csv(filepath + filename)
    print('Saved bgcArgo test data to ' + filepath + filename)

    # Ship data
    filename = 'socat_trainval_processed_co2_mld_adt_yr2014-2023_acc' + datetag + '.csv'
    shipDF_processed.to_csv(filepath + filename)
    print('Saved socat training/validation data to ' + filepath + filename)

print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


Saved bgcArgo training/validation data to ../working-vars/regression/P1-processed/bgcArgo_trainval_processed_co2_mld_adt_soccom20m_pCO2_pHbias5_pK1_yr2014-2023_acc20260211.csv
Saved bgcArgo test data to ../working-vars/regression/P1-processed/bgcArgo_test_processed_co2_mld_adt_soccom20m_yr2024_acc20260211.csv
Saved socat training/validation data to ../working-vars/regression/P1-processed/socat_trainval_processed_co2_mld_adt_yr2014-2023_acc20260211.csv
2026-02-11 14:14:58


In [62]:
# # ==OLD= Optional save: 
# save = True
# datetag = datetime.now().strftime('%Y%m%d') 

# if save:
#     filepath = '../working-vars/regression/P1-processed/'
# #     filepath = '../working-vars/regression/inputs/' + pcm_params + '/' # old jan 2026

#     # Floats
#     filename = 'floatDF_processed_atmos_ADT_SLA_soccom20m_pCO2_pHbias5_pK1_PCM_' + pcm_params + '_acc' + datetag + '.csv'
#     # use_cols = ['profid', 'wmoid', 'prof_datetag', 'cluster', 
#     #           'datetime', 'latitude', 'longitude', 
#     #           'linear_time', 'ydcos', 'ydsin', 'decimalyr', 
#     #           'CT', 'SA', 'mld', 
#     #           'pCO2_standard', 'pCO2_pHbias3', 'pCO2_pHbias5', 'pCO2_pHbias3_pK1', 'pCO2_pHbias5_pK1', 
#     #           'pco2', 'pco2_atm', 'delta_pco2',
#     #           'year', 'month', 'day', 
#     #           'adt', 'sla']
#     use_cols = ['wmoid', 'profid', 'prof_datetag', 'cluster', 
#                 'datetime', 'year', 'month', 'linear_time', 'ydcos', 'ydsin', 
#                 'latitude', 'longitude', 
#                 'CT', 'SA', 'mld', 'adt', 'sst', 'sss',
#                 'atmos_pres_Pa', 'atmos_pres_atm', 'vapor_pres_atm', 
#                 'pCO2_standard', 'pCO2_pHbias3', 'pCO2_pHbias5', 'pCO2_pHbias3_pK1', 'pCO2_pHbias5_pK1',
#                 'pco2_atmos', 'pco2_ocean', 'delta_pco2', 
#                 ]
#     floatDF_added[use_cols].to_csv(filepath + filename)
#     print('Saved floatDF to ' + filepath + filename)

#     # Ship data
#     filename = 'shipDF_processed_atmos_ADT_SLA_socatv2024_1d_pCO2converted_co2sys_PCM_' + pcm_params + '_acc' + datetag + '.csv'
#     use_cols = ['cruiseid', 'sampleid', 'cluster',
#             'datetime', 'year', 'month', 'linear_time', 'ydcos', 'ydsin', 
#             'latitude', 'longitude', 
#             'CT', 'SA', 'pressure', 'mld', 'adt', 'sst', 'sss',
#             'nearest_profid', 'prof_datetime', 'prof_lat', 'prof_lon', 'yd_sep', 'km_sep', 
#             'atmos_pres_Pa', 'atmos_pres_atm', 'vapor_pres_atm', 
#             'fco2rec',
#             'pco2_atmos', 'pco2_ocean', 'delta_pco2']
#     shipDF_added[use_cols].to_csv(filepath + filename)
#     print('Saved shipDF to ' + filepath + filename)
#     print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))



Saved floatDF to ../working-vars/regression/inputs/pc8_gmm7/floatDF_processed_atmos_ADT_SLA_soccom20m_pCO2_pHbias5_pK1_PCM_pc8_gmm7_acc20260210.csv
Saved shipDF to ../working-vars/regression/inputs/pc8_gmm7/shipDF_processed_atmos_ADT_SLA_socatv2024_1d_pCO2converted_co2sys_PCM_pc8_gmm7_acc20260210.csv
2026-02-10 16:19:10


In [99]:
# # EXAMPLE DF FOR OLA
# all_data = pd.concat([floatDF_added, shipDF_added], axis=0, ignore_index=True)

# [floatsave, shipsave] = mod_reg.separate_platforms(all_data)
# shipsave['platform'] = np.tile('socatv2024', len(shipsave))
# floatsave['platform'] = np.tile('bgc', len(floatsave))

# # shipsave['pco2_ocean']
# # floatsave['pco2_ocean'] = floatsave['pCO2_pHbias5_pK1'].copy()

# use_cols = ['latitude', 'longitude', 'datetime', 
#  'pco2_atmos', 'pco2_ocean', 
#  'platform', 'profid', 'wmoid', 'cruiseid', 'sampleid']

# newsave = pd.concat([shipsave, floatsave], axis=0).reset_index()[use_cols]
# newsave.to_csv('../working-vars/regression/exampleDF_bgc_socat_pco2_2014-2023.csv')

# # newsave
# pd.read_csv('../working-vars/regression/exampleDF_bgc_socat_pco2_2014-2023.csv', index_col=0)

## save xr datasets into working-vars/L4-datasets/

In [107]:
bgcDS

<xarray.Dataset>
Dimensions:      (profid: 11782, pressure: 197)
Coordinates:
  * profid       (profid) object '2903453_id001' ... '6903026_id097'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 ...
    latitude     (profid, pressure) float64 ...
    longitude    (profid, pressure) float64 ...
    wmoid        (profid, pressure) float64 ...
    datetime     (profid, pressure) datetime64[ns] ...
Data variables:
    CT           (profid, pressure) float64 ...
    SA           (profid, pressure) float64 ...
    sigma0       (profid, pressure) float64 ...
    spice        (profid, pressure) float64 ...
    temperature  (profid, pressure) float64 ...
    salinity     (profid, pressure) float64 ...
    pH           (profid, pressure) float64 ...
Attributes:
    title:            BGC float profiles with valid data, interpolated to n=2...
    pressure_levels:  Pressure levels in dbar: [0, 5, 10, 15, 20, 25, 30, 35,...
    source:           Argopy, expert mode
    date:             20250729

In [108]:
# Save xarray dataset version also
bgcINDEX_withRegressionVars = floatDF_processed.set_index(['profid']).to_xarray()
bgcINDEX_withRegressionVars

folder = '../working-vars/L4-datasets/'
datetag = datetime.now().strftime('%Y%m%d')

bgcINDEX_withRegressionVars.to_netcdf(folder + 'bgcINDEX_processed_2014-2023_fullco2_mld_adt_acc' + datetag + '.nc')
print('Saved processed bgcINDEX to ' + folder + 'bgcINDEX_processed_2014-2023_fullco2_mld_adt_acc' + datetag + '.nc')
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Saved processed bgcINDEX to ../working-vars/L4-datasets/bgcINDEX_processed_2014-2023_fullco2_mld_adt_acc20260211.nc
2026-02-11 14:36:48


In [112]:
testDF_processed

,wmoid,profid,prof_datetag,datetime,latitude,longitude,ydcos,ydsin,CT,SA,...,atmos_pres_atm,vapor_pres_atm,pCO2_standard,pCO2_pHbias3,pCO2_pHbias5,pCO2_pHbias3_pK1,pCO2_pHbias5_pK1,pco2_atmos,pco2_ocean,delta_pco2
10821,1902494,1902494_testid005,1902494_20240414,2024-04-14 01:05:59.999995,-46.051,8.418,-0.208893,0.977938,8.362248,33.980043,...,0.997977,0.010632,385.186597,382.174335,380.178073,374.436772,372.484853,412.286438,372.484853,-39.801585
10822,1902494,1902494_testid006,1902494_20240415,2024-04-15 00:34:59.999996,-46.079,8.489,-0.225324,0.974284,8.462328,33.977941,...,0.996990,0.010704,390.867094,387.813215,385.789368,379.953823,377.974947,411.843348,377.974947,-33.868401
10823,1902494,1902494_testid006,1902494_20240415,2024-04-15 23:15:59.999996,-46.119,8.556,-0.241134,0.970492,8.386343,33.973935,...,0.999457,0.010649,390.203562,387.155400,385.135341,379.307806,377.332637,412.895518,377.332637,-35.562880
10824,1902494,1902494_testid008,1902494_20240416,2024-04-16 21:53:00.000002,-46.144,8.612,-0.256834,0.966456,8.339002,33.972940,...,1.001036,0.010615,387.736860,384.706857,382.698834,376.911902,374.948496,413.568473,374.948496,-38.619977
10825,1902494,1902494_testid009,1902494_20240417,2024-04-17 20:03:00.000003,-46.172,8.652,-0.272156,0.962253,8.574900,33.975963,...,1.000345,0.010786,390.031218,386.981940,384.961146,379.144770,377.168868,413.207793,377.168868,-36.038925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12929,7902134,7902134_testid001,7902134_20241202,2024-12-02 05:03:00.000003,-43.610,266.023,0.873645,-0.486564,11.344387,34.209100,...,0.997681,0.012986,421.150480,417.841200,415.648130,409.427093,407.282639,411.264266,407.282639,-3.981627
12930,7902134,7902134_testid002,7902134_20241212,2024-12-12 10:44:59.999997,-43.667,266.415,0.945378,-0.325975,12.071960,34.221409,...,1.004392,0.013626,460.583094,456.982503,454.596343,447.729779,445.396625,413.797331,445.396625,31.599294
12931,7902134,7902134_testid003,7902134_20241222,2024-12-22 16:30:59.999996,-43.583,266.845,0.987875,-0.155254,12.017707,34.200234,...,1.005774,0.013577,474.052716,470.357643,467.908847,460.804132,458.409796,414.398989,458.409796,44.010807
12932,7902133,7902133_testid002,7902133_20241207,2024-12-07 07:07:59.999998,-49.189,250.268,0.912825,-0.408351,8.931264,34.281195,...,0.963731,0.011051,456.924390,453.393309,451.053139,444.097836,441.809882,397.741042,441.809882,44.068841


In [113]:
# Save xarray dataset version also
testINDEX_withRegressionVars = testDF_processed.set_index(['profid']).to_xarray()
testINDEX_withRegressionVars

folder = '../working-vars/L4-datasets/'
datetag = datetime.now().strftime('%Y%m%d')

testINDEX_withRegressionVars.to_netcdf(folder + 'testINDEX_processed_2024_fullco2_mld_adt_acc' + datetag + '.nc')
print('Saved processed testINDEX to ' + folder + 'testINDEX_processed_2024_fullco2_mld_adt_acc' + datetag + '.nc')
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Saved processed testINDEX to ../working-vars/L4-datasets/testINDEX_processed_2024_fullco2_mld_adt_acc20260211.nc
2026-02-11 14:52:02


# Application: Add ADT/SLA to Core Argo (LONG RUN TIME)


- Warning: long run time, ~18 hours
- Run already saved in ```../working-vars/regression/inputs/core/coreDS_yrs2014-2023_ADT_SLA_processed_acc20260201.nc```

In [11]:
# coreDF = coreDS.to_dataframe().reset_index().copy()

# coreDF['linear_time'] = coreDF.datetime.apply(lambda x: mod_ocean.datetime2ytd(np.datetime64(x), ref_time='2014-01-01'))
# coreDF['ydcos']= mod_ocean.get_ydsines(coreDF.linear_time.values)[0]
# coreDF['ydsin']= mod_ocean.get_ydsines(coreDF.linear_time.values)[1]
# coreDF = mod_ocean.add_decimalyr(coreDF)


# reload(mod_prep)
# mod_prep.add_regression_vars(coreDF, timeOnly=True)

/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_ocean.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['datetime'].dt.is_leap_year.replace({True: 366, False: 365}))


In [ ]:
coreDF = coreDS.to_dataframe().reset_index().copy()

coreDF['linear_time'] = coreDF.datetime.apply(lambda x: mod_ocean.datetime2ytd(np.datetime64(x), ref_time='2014-01-01'))
coreDF['ydcos']= mod_ocean.get_ydsines(coreDF.linear_time.values)[0]
coreDF['ydsin']= mod_ocean.get_ydsines(coreDF.linear_time.values)[1]
coreDF = mod_ocean.add_decimalyr(coreDF)

coreDF =  mod_ocean.expand_datetime(coreDF, type='dataframe')



In [34]:
from tqdm import tqdm

In [ ]:
# ==== COMMENT OUT TO KEEP OLD PROGRESS
# coreDF_annual = {k:None for k in range(2014,2024)} # store by year


# === Optional save: 
save = True
datetag = datetime.now().strftime('%Y%m%d') 
filepath = '../working-vars/regression/inputs/core/'


# Run over all ten years
for yr in tqdm(range(2015,2024)):
    print('==> Processing year ' + str(yr))
    adt_data = adt_dict[yr]
    core_data = coreDF[coreDF.year==yr].copy()
    core_data['adt'] = np.tile(np.nan, len(core_data))
    # core_data['sla'] = np.tile(np.nan, len(core_data))

    for ind, row in core_data.iterrows():
        closest_row = adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest')
        adtval = closest_row.adt.values.tolist()
        slaval = closest_row.sla.values.tolist()
        core_data.loc[ind, 'adt'] = adtval
        # core_data.loc[ind, 'sla'] = slaval
   
    coreDF_annual[yr] = core_data

    if save:
        filename = 'coreDF_yr' + str(yr)+ '_ADT_SLA_acc' + datetag + '.csv'
        coreDF_annual[yr].to_csv(filepath + filename)
        print('Saved year ' + str(yr) + ' to ' + filename)



coreDF_added = pd.concat(coreDF_annual.values(), axis=0)

  0%|          | 0/9 [00:00<?, ?it/s]

==> Processing year 2015


 11%|█         | 1/9 [1:26:06<11:28:51, 5166.46s/it]

Saved year 2015 to coreDF_yr2015_ADT_SLA_acc20260130.csv
==> Processing year 2016


 22%|██▏       | 2/9 [2:58:19<10:27:53, 5381.94s/it]

...

Saved year 2021 to coreDF_yr2021_ADT_SLA_acc20260130.csv
==> Processing year 2022


 89%|████████▉ | 8/9 [11:36:27<1:23:26, 5006.55s/it]

Saved year 2022 to coreDF_yr2022_ADT_SLA_acc20260130.csv
==> Processing year 2023


100%|██████████| 9/9 [13:07:21<00:00, 5249.09s/it]  

Saved year 2023 to coreDF_yr2023_ADT_SLA_acc20260130.csv


In [ ]:
# === Optional save of full DF : (or save Dataset below)
save = True
datetag = datetime.now().strftime('%Y%m%d') 

if save:
    filepath = '../working-vars/regression/inputs/core/'
    # Floats
    filename = 'coreDF_yrs2014-2023_ADT_SLA_processed_acc' + datetag + '.csv'
    coreDF_added.to_csv(filepath + filename)
    print('Saved coreDF to ' + filepath + filename)

Saved coreDF to ../working-vars/regression/inputs/core/coreDF_yrs2014-2023_ADT_SLA_processed_acc20260131.csv


In [43]:
coreDS

<xarray.Dataset>
Dimensions:      (profid: 328936, pressure: 197)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 1.107 1.107 ... 3.643e+03 3.643e+03
    latitude     (profid, pressure) float64 -40.36 -40.36 ... -51.79 -51.79
    longitude    (profid, pressure) float64 95.36 95.36 95.36 ... -46.82 -46.82
    wmoid        (profid, pressure) float64 1.9e+06 1.9e+06 ... 7.901e+06
    datetime     (profid, pressure) datetime64[ns] 2014-01-02T02:34:12 ... 20...
Data variables:
    CT           (profid, pressure) float64 12.71 12.71 12.7 ... 2.183 2.176
    SA           (profid, pressure) float64 35.03 35.03 35.03 ... 34.8 34.8 34.8
    sigma0       (profid, pressure) float64 26.35 26.35 26.35 ... 27.67 27.67
    spice        (profid, pressure) float64 1.69 1.69 1.688 ... -0.08716 -0.0865
    temperature  (profid, pressure) float64 12.72 12.72 12.71 ... 2.244 2.237
    salinity     (profid, pressure) float64 34.87 34.87 34.87 ... 34.63 34.63
    mld          (profid) float64 90.55 66.09 13.02 74.86 ... 82.53 71.18 19.52
    mlp          (profid) object 91.27 66.61 13.12 75.46 ... 83.29 71.83 19.69
Attributes:
    title:            Core float profiles with valid surface data (QC flags 1...
    pressure_levels:  Pressure levels in dbar: [0, 5, 10, 15, 20, 25, 30, 35,...
    source:           Argopy, expert mode; GDAC snapshot Jan 2025
    date:             20250424

In [ ]:
coreDS_added = mod_argo.to_xr_dataset(coreDF_added)
temp = xr.Dataset.from_dataframe(coreDF_added.set_index(["profid", 'pressure']))
nc_attrs={'date':str(datetime.now()),
          'adt_type': "Global Ocean Gridded L 4 Sea Surface Heights And Derived Variables Reprocessed 1993 Ongoing",
          'adt_source': "https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_L4_MY_008_047/description"}

argo_DS = temp.set_coords(['profid', 'datetime', 'yearday', 'latitude', 'longitude'])
argo_DS = argo_DS.assign_attrs(nc_attrs)
argo_DS

<xarray.Dataset>
Dimensions:      (profid: 328936, pressure: 197)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 1.107 1.107 ... 3.643e+03 3.643e+03
    latitude     (profid, pressure) float64 -40.36 -40.36 ... -51.79 -51.79
    longitude    (profid, pressure) float64 95.36 95.36 95.36 ... -46.82 -46.82
    datetime     (profid, pressure) datetime64[ns] 2014-01-02T02:34:12 ... 20...
Data variables: (12/18)
    CT           (profid, pressure) float64 12.71 12.71 12.7 ... 2.183 2.176
    SA           (profid, pressure) float64 35.03 35.03 35.03 ... 34.8 34.8 34.8
    sigma0       (profid, pressure) float64 26.35 26.35 26.35 ... 27.67 27.67
    spice        (profid, pressure) float64 1.69 1.69 1.688 ... -0.08716 -0.0865
    temperature  (profid, pressure) float64 12.72 12.72 12.71 ... 2.244 2.237
    salinity     (profid, pressure) float64 34.87 34.87 34.87 ... 34.63 34.63
    ...           ...
    decimalyr    (profid, pressure) float64 2.014e+03 2.014e+03 ... 2.024e+03
    year         (profid, pressure) int64 2014 2014 2014 2014 ... 2023 2023 2023
    month        (profid, pressure) int64 1 1 1 1 1 1 1 ... 12 12 12 12 12 12 12
    day          (profid, pressure) int64 2 2 2 2 2 2 2 ... 23 23 23 23 23 23 23
    adt          (profid, pressure) float64 0.5323 0.5323 ... -0.5156 -0.5156
    sla          (profid, pressure) float64 -0.012 -0.012 ... -0.0052 -0.0052
Attributes:
    date:        2026-02-01 12:29:20.321568
    adt_type:    Global Ocean Gridded L 4 Sea Surface Heights And Derived Var...
    adt_source:  https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_L...

In [ ]:
# === Optional save of full xr Dataset: 
save = True
datetag = datetime.now().strftime('%Y%m%d') 

if save:
    # filepath = '../working-vars/regression/inputs/core/'

    filepath = '../working-vars/L4-datasets/' # new location feb 10 2026
    # Floats
    filename = 'coreDATA_processed_2014-2023_MLD_ADT_SLA_acc' + datetag + '.nc'
    argo_DS.to_netcdf(filepath + filename)
    print('Saved coreDS to ' + filepath + filename)

    # later accessed through loader.import_core_data(L3_only = False)

Saved coreDS to ../working-vars/regression/inputs/core/coreDS_yrs2014-2023_ADT_SLA_processed_acc20260201.nc


In [52]:
argo_DS

<xarray.Dataset>
Dimensions:      (profid: 328936, pressure: 197)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 1.107 1.107 ... 3.643e+03 3.643e+03
    latitude     (profid, pressure) float64 -40.36 -40.36 ... -51.79 -51.79
    longitude    (profid, pressure) float64 95.36 95.36 95.36 ... -46.82 -46.82
    datetime     (profid, pressure) datetime64[ns] 2014-01-02T02:34:12 ... 20...
Data variables: (12/18)
    CT           (profid, pressure) float64 12.71 12.71 12.7 ... 2.183 2.176
    SA           (profid, pressure) float64 35.03 35.03 35.03 ... 34.8 34.8 34.8
    sigma0       (profid, pressure) float64 26.35 26.35 26.35 ... 27.67 27.67
    spice        (profid, pressure) float64 1.69 1.69 1.688 ... -0.08716 -0.0865
    temperature  (profid, pressure) float64 12.72 12.72 12.71 ... 2.244 2.237
    salinity     (profid, pressure) float64 34.87 34.87 34.87 ... 34.63 34.63
    ...           ...
    decimalyr    (profid, pressure) float64 2.014e+03 2.014e+03 ... 2.024e+03
    year         (profid, pressure) int64 2014 2014 2014 2014 ... 2023 2023 2023
    month        (profid, pressure) int64 1 1 1 1 1 1 1 ... 12 12 12 12 12 12 12
    day          (profid, pressure) int64 2 2 2 2 2 2 2 ... 23 23 23 23 23 23 23
    adt          (profid, pressure) float64 0.5323 0.5323 ... -0.5156 -0.5156
    sla          (profid, pressure) float64 -0.012 -0.012 ... -0.0052 -0.0052
Attributes:
    date:        2026-02-01 12:29:20.321568
    adt_type:    Global Ocean Gridded L 4 Sea Surface Heights And Derived Var...
    adt_source:  https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_L...

# Application: Finish prep Core Argo index (add atmospheric carbon)

In [75]:
# [coreDS, coreINDEX] = loader.import_core_data(type='processed')
# coreINDEX

<xarray.Dataset>
Dimensions:         (profid: 328938)
Coordinates:
  * profid          (profid) object '1900410_id260' ... '7901011_id005'
Data variables: (12/32)
    pressure        (profid) float64 ...
    wmoid           (profid) float64 ...
    latitude        (profid) float64 ...
    longitude       (profid) float64 ...
    datetime        (profid) datetime64[ns] ...
    yearday         (profid) float64 ...
    ...              ...
    atmos_pres_atm  (profid) float64 ...
    atmos_co2_ppm   (profid) float64 ...
    sst             (profid) float64 ...
    sss             (profid) float64 ...
    vapor_pres_atm  (profid) float64 ...
    pco2_atmos      (profid) float64 ...

In [ ]:
# PROCESSED CORE ARGO DATAFRAME WITH REGRESSION VARS ADDED 
# ~30 min run for all core
coreind_df = coreINDEX.to_dataframe().reset_index().copy()
coreind_df.head()

coreind_df = mod_prep.add_regression_time_vars(coreind_df)

coreind_df['lon_round'] = round((coreind_df['longitude'] + 180)/2.5)*2.5
coreind_df['lat_round'] = round(coreind_df['latitude']/2.5)*2.5

coreind_df = mod_prep.add_regression_carbon_vars(coreind_df, convert_pco2=True) 
# coreind_df['delta_pco2'] = coreind_df['pco2_ocean'] - coreind_df['pco2_atmos']

In [ ]:
coreINDEX

In [97]:
coreind_df

,profid,pressure,wmoid,latitude,longitude,datetime,yearday,CT,SA,sigma0,...,pco2_atm,lon_round,lat_round,atmos_pres_Pa,atmos_pres_atm,atmos_co2_ppm,sst,sss,vapor_pres_atm,pco2_atmos
0,1900410_id260,6.4,1900410.0,-40.358000,95.357000,2014-01-02 02:34:12.000000000,1.107083,12.707135,35.034611,26.352269,...,393.891511,275.0,-40.0,102290.0,1.009524,393.891511,12.715140,35.199985,0.014217,392.043076
1,1900410_id261,7.0,1900410.0,-40.109000,96.083000,2014-01-12 06:35:16.000000000,11.274491,13.136123,35.033602,26.265798,...,393.899578,275.0,-40.0,102080.0,1.007451,393.899578,13.144039,35.198965,0.014621,391.075263
2,1900410_id262,6.0,1900410.0,-40.247000,96.676000,2014-01-22 10:36:20.000000000,21.441898,14.351957,34.975328,25.968933,...,393.883802,277.5,-40.0,100920.0,0.996003,393.883802,14.358125,35.140420,0.015823,386.076861
3,1900410_id263,6.6,1900410.0,-40.371000,96.900000,2014-02-01 14:33:32.000000000,31.606620,13.675696,35.015520,26.141685,...,393.863153,277.5,-40.0,101600.0,1.002714,393.863153,13.683068,35.180803,0.015145,388.967172
4,1900410_id264,5.9,1900410.0,-40.340000,96.969000,2014-02-11 18:47:10.000000000,41.782755,13.617688,35.019535,26.156743,...,393.846194,277.5,-40.0,102700.0,1.013570,393.846194,13.625170,35.184837,0.015088,393.248530
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328933,7901011_id001,10.6,7901011.0,-54.642010,-51.690847,2023-11-11 21:57:00.000000000,3601.914583,3.979210,34.266818,27.078063,...,417.746218,127.5,-55.0,99170.0,0.978732,417.746218,3.975286,34.429393,0.007856,405.579794
328934,7901011_id002,3.1,7901011.0,-53.820388,-49.906383,2023-11-23 17:01:30.000000000,3613.709375,4.143979,34.242454,27.041822,...,417.684612,130.0,-55.0,100730.0,0.994128,417.684612,4.139787,34.404664,0.007947,411.912454
328935,7901011_id003,2.8,7901011.0,-53.981737,-48.127877,2023-12-03 12:41:30.000000000,3623.528819,3.454201,34.212383,27.086921,...,417.642774,132.5,-55.0,99240.0,0.979423,417.642774,3.449824,34.374525,0.007570,405.887082
328936,7901011_id004,3.1,7901011.0,-52.839405,-48.947265,2023-12-13 07:27:30.000000000,3633.310764,3.677531,34.179080,27.038900,...,417.536717,130.0,-52.5,99600.0,0.982976,417.536717,3.672800,34.340917,0.007691,407.217298


In [ ]:
# coreind_df.to_csv('../working-vars/regression/P1-processed/coreArgo_application_processed_co2_mld_adt_yr2014-2023_acc' + datetag + '.csv')

In [81]:
coreINDEX #coreind_df.to_csv('../working-vars/regression/temp_coreind_newpco2.csv')

<xarray.Dataset>
Dimensions:      (profid: 328938)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
Data variables: (12/23)
    pressure     (profid) float64 6.4 7.0 6.0 6.6 5.9 ... 10.6 3.1 2.8 3.1 2.9
    wmoid        (profid) float64 1.9e+06 1.9e+06 ... 7.901e+06 7.901e+06
    latitude     (profid) float64 -40.36 -40.11 -40.25 ... -53.98 -52.84 -51.79
    longitude    (profid) float64 95.36 96.08 96.68 ... -48.13 -48.95 -46.82
    datetime     (profid) datetime64[ns] 2014-01-02T02:34:12 ... 2023-12-23T0...
    yearday      (profid) float64 1.107 11.27 21.44 ... 3.633e+03 3.643e+03
    ...           ...
    ydcos        (profid) float64 0.9998 0.9813 0.9327 ... 0.8784 0.946 0.987
    ydsin        (profid) float64 0.01904 0.1927 0.3605 ... -0.3241 -0.1608
    linear_time  (profid) float64 1.107 11.27 21.44 ... 3.633e+03 3.643e+03
    decimalyr    (profid) float64 2.014e+03 2.014e+03 ... 2.024e+03 2.024e+03
    sin_lat      (profid) float64 -0.6476 -0.6442 -0.6461 ... -0.7969 -0.7857
    pco2_atm     (profid) float64 393.9 393.9 393.9 393.9 ... 417.6 417.5 417.5

In [101]:
# Save xarray dataset version also
coreINDEX_withRegressionVars = coreind_df.set_index(['profid']).to_xarray()
coreINDEX_withRegressionVars.drop_vars(['temperature', 'salinity'])
coreINDEX_withRegressionVars


# folder = '../working-vars/regression/inputs/core/'
# coreINDEX_withRegressionVars.to_netcdf(folder + 'coreINDEX_pco2-atmos_processed_2014-2023_MLD_ADT_acc' + datetag + '.nc')

folder = '../working-vars/L4-datasets/'
datetag = datetime.now().strftime('%Y%m%d')

coreINDEX_withRegressionVars.to_netcdf(folder + 'coreINDEX_processed_2014-2023_co2_mld_adt_acc' + datetag + '.nc')
print('Saved processed coreINDEX to ' + folder + 'coreINDEX_processed_2014-2023_co2_mld_adt_acc' + datetag + '.nc')
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Saved processed coreINDEX to ../working-vars/L4-datasets/coreINDEX_processed_2014-2023_co2_mld_adt_acc20260211.nc
2026-02-11 14:23:39


In [102]:
coreINDEX_withRegressionVars

<xarray.Dataset>
Dimensions:         (profid: 328938)
Coordinates:
  * profid          (profid) object '1900410_id260' ... '7901011_id005'
Data variables: (12/32)
    pressure        (profid) float64 6.4 7.0 6.0 6.6 5.9 ... 3.1 2.8 3.1 2.9
    wmoid           (profid) float64 1.9e+06 1.9e+06 ... 7.901e+06 7.901e+06
    latitude        (profid) float64 -40.36 -40.11 -40.25 ... -52.84 -51.79
    longitude       (profid) float64 95.36 96.08 96.68 ... -48.13 -48.95 -46.82
    datetime        (profid) object '2014-01-02 02:34:12.000000000' ... '2023...
    yearday         (profid) float64 1.107 11.27 21.44 ... 3.633e+03 3.643e+03
    ...              ...
    atmos_pres_atm  (profid) float64 1.01 1.007 0.996 ... 0.9794 0.983 0.9868
    atmos_co2_ppm   (profid) float64 393.9 393.9 393.9 ... 417.6 417.5 417.5
    sst             (profid) float64 12.72 13.14 14.36 ... 3.45 3.673 4.811
    sss             (profid) float64 35.2 35.2 35.14 35.18 ... 34.37 34.34 34.32
    vapor_pres_atm  (profid) float64 0.01422 0.01462 ... 0.007691 0.00833
    pco2_atmos      (profid) float64 392.0 391.1 386.1 ... 405.9 407.2 408.5

# CONSTRUCT EXTENDED CORE ARGO DATASET FOR CLUSTERING

- going to do this in 2.1_pcm_fit_coreArgo

Combine 2014-2023 core argo + 2024 bgcDS profiles which also have CT, SA

- After clustering you should output 'core'INDEX_test_clustered as well as coreINDEX_test_clustered


        - technically same as bgcINDEX_test_clustered but if you want to keep consistent/in same folder

In [ ]:
[coreDS, coreINDEX] = loader.import_core_data(type='processed') # has adt, sla, mld, pco2
[bgcDS_test, bgcINDEX_test] = loader.import_bgc_data(type='test')
bgcDS_test

In [84]:
# testDF_processed['profid'] = ([x.replace('id', 'testid') for x in testDF_processed['profid']])
# testDF_processed

/var/folders/nt/sjynqxjj7cz9r15fkd5d4r_40000gp/T/ipykernel_87096/2251034600.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  testDF_processed['profid'] = ([x.replace('id', 'testid') for x in testDF_processed['profid']])


,wmoid,profid,prof_datetag,datetime,latitude,longitude,ydcos,ydsin,CT,SA,...,atmos_pres_atm,vapor_pres_atm,pCO2_standard,pCO2_pHbias3,pCO2_pHbias5,pCO2_pHbias3_pK1,pCO2_pHbias5_pK1,pco2_atmos,pco2_ocean,delta_pco2
10821,1902494,1902494_testid005,1902494_20240414,2024-04-14 01:05:59.999995,-46.051,8.418,-0.208893,0.977938,8.362248,33.980043,...,0.997977,0.010632,385.186597,382.174335,380.178073,374.436772,372.484853,412.286438,372.484853,-39.801585
10822,1902494,1902494_testid006,1902494_20240415,2024-04-15 00:34:59.999996,-46.079,8.489,-0.225324,0.974284,8.462328,33.977941,...,0.996990,0.010704,390.867094,387.813215,385.789368,379.953823,377.974947,411.843348,377.974947,-33.868401
10823,1902494,1902494_testid006,1902494_20240415,2024-04-15 23:15:59.999996,-46.119,8.556,-0.241134,0.970492,8.386343,33.973935,...,0.999457,0.010649,390.203562,387.155400,385.135341,379.307806,377.332637,412.895518,377.332637,-35.562880
10824,1902494,1902494_testid008,1902494_20240416,2024-04-16 21:53:00.000002,-46.144,8.612,-0.256834,0.966456,8.339002,33.972940,...,1.001036,0.010615,387.736860,384.706857,382.698834,376.911902,374.948496,413.568473,374.948496,-38.619977
10825,1902494,1902494_testid009,1902494_20240417,2024-04-17 20:03:00.000003,-46.172,8.652,-0.272156,0.962253,8.574900,33.975963,...,1.000345,0.010786,390.031218,386.981940,384.961146,379.144770,377.168868,413.207793,377.168868,-36.038925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12929,7902134,7902134_testid001,7902134_20241202,2024-12-02 05:03:00.000003,-43.610,266.023,0.873645,-0.486564,11.344387,34.209100,...,0.997681,0.012986,421.150480,417.841200,415.648130,409.427093,407.282639,411.264266,407.282639,-3.981627
12930,7902134,7902134_testid002,7902134_20241212,2024-12-12 10:44:59.999997,-43.667,266.415,0.945378,-0.325975,12.071960,34.221409,...,1.004392,0.013626,460.583094,456.982503,454.596343,447.729779,445.396625,413.797331,445.396625,31.599294
12931,7902134,7902134_testid003,7902134_20241222,2024-12-22 16:30:59.999996,-43.583,266.845,0.987875,-0.155254,12.017707,34.200234,...,1.005774,0.013577,474.052716,470.357643,467.908847,460.804132,458.409796,414.398989,458.409796,44.010807
12932,7902133,7902133_testid002,7902133_20241207,2024-12-07 07:07:59.999998,-49.189,250.268,0.912825,-0.408351,8.931264,34.281195,...,0.963731,0.011051,456.924390,453.393309,451.053139,444.097836,441.809882,397.741042,441.809882,44.068841


In [4]:
[bgcDS_test, bgcINDEX_test] = loader.import_bgc_data(type='test')
bgcDS_test

<xarray.Dataset>
Dimensions:      (profid: 2387, pressure: 197)
Coordinates:
  * profid       (profid) object '1902491_id001' ... '7902134_id003'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 ...
    latitude     (profid, pressure) float64 ...
    longitude    (profid, pressure) float64 ...
    datetime     (profid, pressure) datetime64[ns] ...
Data variables:
    CT           (profid, pressure) float64 ...
    SA           (profid, pressure) float64 ...
    sigma0       (profid, pressure) float64 ...
    spice        (profid, pressure) float64 ...
    temperature  (profid, pressure) float64 ...
    salinity     (profid, pressure) float64 ...
    pH           (profid, pressure) float64 ...
    wmoid        (profid, pressure) float64 ...
Attributes:
    title:            BGC float profiles with valid data, interpolated to n=2...
    pressure_levels:  Pressure levels in dbar: [0, 5, 10, 15, 20, 25, 30, 35,...
    source:           Argopy, expert mode
    date:             20260209

In [9]:
# bgcDS_test['profid'] = bgcDS_test['profid'].apply(lambda x: x.replace('id', 'testid'))

temp = bgcDS_test.to_dataframe().reset_index().copy()
temp['profid'] = temp['profid'].apply(lambda x: x.replace('id', 'testid'))
bgcDS_test_updated = temp.set_index(['profid', 'pressure']).to_xarray()
# bgcDS_test_updated.assign_coords()

ValueError: the first argument to .assign_coords must be a dictionary

In [120]:
bgcINDEX_test

<xarray.Dataset>
Dimensions:           (profid: 2113)
Coordinates:
  * profid            (profid) object '1902494_testid005' ... '7902133_testid...
Data variables: (12/24)
    wmoid             (profid) object ...
    prof_datetag      (profid) object ...
    datetime          (profid) datetime64[ns] ...
    latitude          (profid) float64 ...
    longitude         (profid) float64 ...
    ydcos             (profid) float64 ...
    ...                ...
    pCO2_pHbias5      (profid) float64 ...
    pCO2_pHbias3_pK1  (profid) float64 ...
    pCO2_pHbias5_pK1  (profid) float64 ...
    pco2_atmos        (profid) float64 ...
    pco2_ocean        (profid) float64 ...
    delta_pco2        (profid) float64 ...